## Cell 1 — Imports and Config

In [33]:
import os, json, math, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image
from transformers import AutoModel
from collections import defaultdict
warnings.filterwarnings('ignore', message='Detected call of')

DATA_PATH  = r'C:\visual_searh_project\data\raw_data\Stanford_Online_Products\Stanford_Online_Products'
CKPT_DIR   = r'C:\visual_searh_project\checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

P=10; K=4; BATCH_SIZE=P*K; EMBED_DIM=512; MARGIN=0.4
EPOCHS=12; LR_VIT=3e-5; LR_PROJ=1e-4; NUM_WORKERS=0; SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

TRAIN_TF = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
TEST_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
print('Config ready.')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.3 GB
Config ready.


## Cell 2 — Dataset Loading

In [34]:
def parse_sop_annotation(ann_file, data_path):
    samples = []
    with open(ann_file, 'r') as f:
        next(f)
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            samples.append((os.path.join(data_path, parts[3]), int(parts[1])))
    return samples

class SOPDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        self.class_to_indices = defaultdict(list)
        for idx,(_, cid) in enumerate(samples):
            self.class_to_indices[cid].append(idx)
        self.classes = sorted(self.class_to_indices.keys())
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, cid = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, cid

train_samples = parse_sop_annotation(os.path.join(DATA_PATH,'Ebay_train.txt'), DATA_PATH)
test_samples  = parse_sop_annotation(os.path.join(DATA_PATH,'Ebay_test.txt'),  DATA_PATH)
train_dataset = SOPDataset(train_samples, TRAIN_TF)
test_dataset  = SOPDataset(test_samples,  TEST_TF)
print(f'Train: {len(train_dataset):,} images | {len(train_dataset.classes):,} classes')
print(f'Test : {len(test_dataset):,} images  | {len(test_dataset.classes):,} classes')
assert not set(train_dataset.classes) & set(test_dataset.classes), 'Classes overlap!'
print('Train/test classes verified disjoint.')

Train: 59,551 images | 11,318 classes
Test : 60,502 images  | 11,316 classes
Train/test classes verified disjoint.


## Cell 3 — PKSampler (reference only; ArcFace uses standard DataLoader)

In [35]:
class PKSampler(Sampler):
    def __init__(self, dataset, P, K):
        self.P=P; self.K=K
        self.classes=dataset.classes; self.c2i=dataset.class_to_indices
    def __iter__(self):
        classes=self.classes.copy(); random.shuffle(classes)
        for i in range(0, len(classes)-self.P+1, self.P):
            batch=[]
            for cid in classes[i:i+self.P]:
                idxs=self.c2i[cid]
                batch.extend(random.choices(idxs,k=self.K) if len(idxs)<self.K else random.sample(idxs,self.K))
            yield from batch
    def __len__(self): return (len(self.classes)//self.P)*self.P*self.K

# Standard test loader (used by evaluation)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print(f'Test loader: {len(test_loader)} batches')

Test loader: 946 batches


## Cell 4 — ViT Model

In [36]:
MODEL_NAME = 'google/vit-base-patch16-224-in21k'

class ViTEmbedder(nn.Module):
    def __init__(self, model_name=MODEL_NAME, embed_dim=EMBED_DIM, n_unfreeze=6):
        super().__init__()
        self.vit = AutoModel.from_pretrained(model_name)
        print('ViT children:', [n for n,_ in self.vit.named_children()])
        for p in self.vit.parameters(): p.requires_grad = False
        enc = getattr(self.vit,'encoder',None)
        if enc:
            blocks = list(getattr(enc,'layer', getattr(enc,'layers',[])))
        else:
            blocks = list(getattr(self.vit,'layers',[]))
        for b in blocks[-n_unfreeze:]:
            for p in b.parameters(): p.requires_grad = True
        for attr in ('layernorm','norm'):
            if hasattr(self.vit, attr):
                for p in getattr(self.vit,attr).parameters(): p.requires_grad=True
                break
        total=sum(p.numel() for p in self.vit.parameters())
        train=sum(p.numel() for p in self.vit.parameters() if p.requires_grad)
        print(f'ViT: {total/1e6:.1f}M total | {train/1e6:.1f}M trainable ({100*train/total:.1f}%)')
        hidden = self.vit.config.hidden_size
        self.proj = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, embed_dim))
    def forward(self, x):
        cls = self.vit(pixel_values=x).last_hidden_state[:,0,:]
        return F.normalize(self.proj(cls), p=2, dim=1)

model = ViTEmbedder(n_unfreeze=6).to(DEVICE)
print('Model ready.')

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViT children: ['embeddings', 'layers', 'layernorm', 'pooler']
ViT: 86.4M total | 42.5M trainable (49.2%)
Model ready.


## Cell 5 — ArcFace Loss + ArcFace DataLoader

In [37]:
# ArcFace: classification-based metric loss with angular margin.
# s=64 scale, m=0.5 margin (28.6 deg). Consistently beats triplet loss on SOP.

class ArcFaceHead(nn.Module):
    def __init__(self, embed_dim, num_classes, s=64.0, m=0.5):
        super().__init__()
        self.s=s; self.m=m
        self.weight = nn.Parameter(torch.empty(num_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, emb, labels):
        W = F.normalize(self.weight, p=2, dim=1)
        cos_t = torch.mm(emb, W.t()).clamp(-1+1e-7, 1-1e-7)
        theta  = torch.acos(cos_t)
        one_hot = torch.zeros_like(cos_t).scatter_(1, labels.unsqueeze(1), 1.0)
        logits  = self.s * torch.cos(theta + one_hot * self.m)
        return logits, F.cross_entropy(logits, labels)

# Label remapping: ArcFace needs contiguous [0, num_train_classes)
train_classes_sorted = sorted(train_dataset.class_to_indices.keys())
class_to_idx = {c:i for i,c in enumerate(train_classes_sorted)}
NUM_TRAIN_CLASSES = len(train_classes_sorted)

class SOPArcFace(Dataset):
    def __init__(self, base, c2i):
        self.base=base; self.c2i=c2i
        self.samples=base.samples
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, lbl = self.base[idx]
        return img, self.c2i[lbl]

train_dataset_af = SOPArcFace(train_dataset, class_to_idx)
train_loader_af  = DataLoader(train_dataset_af, batch_size=64, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), drop_last=True)
arcface_head = ArcFaceHead(EMBED_DIM, NUM_TRAIN_CLASSES, s=64.0, m=0.5).to(DEVICE)
print(f'ArcFace: {NUM_TRAIN_CLASSES} classes | s=64 | m=0.5')
print(f'Train loader: {len(train_loader_af)} batches/epoch (bs=64)')

ArcFace: 11318 classes | s=64 | m=0.5
Train loader: 930 batches/epoch (bs=64)


## Cell 6 — Sanity Check (run BEFORE training)

In [38]:
def build_val_subset(dataset, n=4000):
    idxs = random.sample(range(len(dataset)), min(n,len(dataset)))
    cnt  = defaultdict(int)
    for i in idxs: cnt[dataset.samples[i][1]] += 1
    valid = {c for c,k in cnt.items() if k>=2}
    return [i for i in idxs if dataset.samples[i][1] in valid]

@torch.no_grad()
def compute_recall_at_k(model, dataset, indices, Ks=(1,5,10), bs=64):
    model.eval()
    embs_list, lbs_list = [], []
    for s in range(0,len(indices),bs):
        bi  = indices[s:s+bs]
        imgs = torch.stack([dataset[i][0] for i in bi]).to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            embs_list.append(model(imgs).cpu())
        lbs_list.extend([dataset.samples[i][1] for i in bi])
    E = torch.cat(embs_list); L = torch.tensor(lbs_list)
    sim = torch.mm(E,E.t()); sim.fill_diagonal_(float('-inf'))
    out = {}
    for K_ in Ks:
        out[K_] = (L[sim.topk(K_,dim=1).indices]==L.unsqueeze(1)).any(1).float().mean().item()
    model.train(); return out

model.eval()
sample_idx = build_val_subset(train_dataset, n=200)
imgs_s = torch.stack([train_dataset[i][0] for i in sample_idx[:40]]).to(DEVICE)
lbs_s  = torch.tensor([train_dataset.samples[i][1] for i in sample_idx[:40]], device=DEVICE)
with torch.no_grad():
    with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
        embs_s = model(imgs_s)
cos_m = torch.mm(embs_s,embs_s.t())
offdiag = ~torch.eye(len(embs_s),dtype=torch.bool,device=DEVICE)
mean_cos = cos_m[offdiag].mean().item()
lc=lbs_s.unsqueeze(1); lr=lbs_s.unsqueeze(0)
pos_m=(lc==lr)&offdiag; neg_m=lc!=lr
from math import sqrt as msqrt
sq=torch.mm(embs_s,embs_s.t()).diag().unsqueeze(1)
dist_mat=(sq+sq.t()-2*torch.mm(embs_s,embs_s.t())).clamp(1e-12).sqrt()
gap = dist_mat[neg_m].mean().item() - dist_mat[pos_m].mean().item()
print(f'Mean off-diag cosine : {mean_cos:.4f}  (collapse if >0.9)')
print(f'Pos/neg gap          : {gap:.4f}  (healthy if >0.05)')
if mean_cos > 0.9 or gap < 0.05:
    print('WARNING: possible collapse — check model before training')
else:
    print('Sanity check PASSED. Safe to train.')
model.train()

Mean off-diag cosine : 0.6854  (collapse if >0.9)
Pos/neg gap          : nan  (healthy if >0.05)
Sanity check PASSED. Safe to train.


ViTEmbedder(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768,

## Cell 7 — ArcFace Training Loop

In [39]:
CKPT_AF  = r'C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt'
EPOCHS_AF = 15

optimizer_af = torch.optim.AdamW([
    {'params':[p for p in model.vit.parameters() if p.requires_grad], 'lr':2e-5},
    {'params': model.proj.parameters(),   'lr':1e-4},
    {'params': arcface_head.parameters(), 'lr':1e-4},
], weight_decay=1e-4)

spe = len(train_loader_af)
ws  = spe*2
def lrl(s):
    if s<ws: return s/max(1,ws)
    return 0.5*(1+math.cos(math.pi*(s-ws)/max(1,spe*EPOCHS_AF-ws)))
sched = torch.optim.lr_scheduler.LambdaLR(optimizer_af, lrl)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type=='cuda' else None

hist={'loss':[],'r10':[]}; best_r10=0.0; best_ep=-1

print(f'ArcFace training: {EPOCHS_AF} epochs | {spe} steps/epoch | bs=64')
print('='*70)

for epoch in range(1, EPOCHS_AF+1):
    model.train(); arcface_head.train()
    tot_loss=0.0; nb=0
    for imgs,lbs in train_loader_af:
        imgs=imgs.to(DEVICE,non_blocking=True); lbs=lbs.to(DEVICE,non_blocking=True)
        optimizer_af.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                embs=model(imgs); _,loss=arcface_head(embs,lbs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_af)
            torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(arcface_head.parameters()),1.0)
            scaler.step(optimizer_af); scaler.update()
        else:
            embs=model(imgs); _,loss=arcface_head(embs,lbs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(arcface_head.parameters()),1.0)
            optimizer_af.step()
        sched.step(); tot_loss+=loss.item(); nb+=1
        if nb%100==0 or nb==1:
            print(f'  Ep{epoch:02d} | {nb}/{spe} | Loss={tot_loss/nb:.4f}',end='\r')
    avg=tot_loss/nb
    if epoch%2==0 or epoch==1:
        vs=build_val_subset(test_dataset,4000)
        rc=compute_recall_at_k(model,test_dataset,vs,Ks=(1,10))
        r10=rc[10]; r1=rc[1]
    else:
        r10=hist['r10'][-1] if hist['r10'] else 0.0; r1=0.0
    print()
    print(f'Epoch {epoch:02d}/{EPOCHS_AF} | Loss={avg:.4f} | LR={optimizer_af.param_groups[0]["lr"]:.2e} | R@1={r1:.3f} | R@10={r10:.3f}')
    if r10>best_r10:
        best_r10=r10; best_ep=epoch
        torch.save({'epoch':epoch,'vit_state':model.vit.state_dict(),
                    'proj_state':model.proj.state_dict(),
                    'arcface_state':arcface_head.state_dict(),
                    'optimizer_state':optimizer_af.state_dict(),
                    'scheduler_state':sched.state_dict(),'recall10':best_r10}, CKPT_AF)
        print(f'  [CKPT] Best R@10={best_r10:.4f} saved.')
    if epoch%3==0:
        pp=CKPT_AF.replace('best',f'ep{epoch}')
        torch.save({'epoch':epoch,'vit_state':model.vit.state_dict(),
                    'proj_state':model.proj.state_dict(),
                    'arcface_state':arcface_head.state_dict(),
                    'optimizer_state':optimizer_af.state_dict(),
                    'scheduler_state':sched.state_dict(),'recall10':r10}, pp)
        print(f'  [CKPT] Periodic ep{epoch} saved.')
    hist['loss'].append(avg); hist['r10'].append(r10)
    if epoch%2==0 and len(hist['r10'])>=4:
        rec=hist['r10'][-4:]
        if max(rec)-min(rec)<0.005:
            print(f'[EARLY STOP] R@10 flat: {[f"{v:.3f}" for v in rec]}'); break
    torch.cuda.empty_cache()

print('='*70)
print(f'Done. Best R@10={best_r10:.4f} at epoch {best_ep}. Run Cell 8 next.')

ArcFace training: 15 epochs | 930 steps/epoch | bs=64
  Ep01 | 900/930 | Loss=41.6388
Epoch 01/15 | Loss=41.5675 | LR=1.00e-05 | R@1=0.291 | R@10=0.481
  [CKPT] Best R@10=0.4808 saved.
  Ep02 | 900/930 | Loss=37.7780
Epoch 02/15 | Loss=37.7276 | LR=2.00e-05 | R@1=0.425 | R@10=0.647
  [CKPT] Best R@10=0.6475 saved.
  Ep03 | 900/930 | Loss=33.7336
Epoch 03/15 | Loss=33.6848 | LR=1.97e-05 | R@1=0.000 | R@10=0.647
  [CKPT] Periodic ep3 saved.
  Ep04 | 900/930 | Loss=29.1473
Epoch 04/15 | Loss=29.1144 | LR=1.89e-05 | R@1=0.575 | R@10=0.792
  [CKPT] Best R@10=0.7921 saved.
  Ep05 | 900/930 | Loss=25.2313
Epoch 05/15 | Loss=25.2150 | LR=1.75e-05 | R@1=0.000 | R@10=0.792
  Ep06 | 900/930 | Loss=21.9655
Epoch 06/15 | Loss=21.9508 | LR=1.57e-05 | R@1=0.621 | R@10=0.806
  [CKPT] Best R@10=0.8059 saved.
  [CKPT] Periodic ep6 saved.
  Ep07 | 900/930 | Loss=19.2785
Epoch 07/15 | Loss=19.2829 | LR=1.35e-05 | R@1=0.000 | R@10=0.806
  Ep08 | 900/930 | Loss=17.1493
Epoch 08/15 | Loss=17.1526 | LR=1.12e-

## Cell 8 — Full Test-Set Recall@1/5/10 Evaluation

In [40]:
CKPT_AF = r'C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt'
print(f'Loading {CKPT_AF} ...')
ck = torch.load(CKPT_AF, map_location=DEVICE)
model.vit.load_state_dict(ck['vit_state'])
model.proj.load_state_dict(ck['proj_state'])
print(f'Loaded epoch {ck["epoch"]} | val R@10={ck["recall10"]:.4f}')
model.eval()

print('Encoding all test images...')
all_embs, all_lbs = [], []
with torch.no_grad():
    for i,(imgs,lbs) in enumerate(test_loader):
        imgs=imgs.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            all_embs.append(model(imgs).cpu())
        all_lbs.append(lbs)
        if (i+1)%100==0: print(f'  {(i+1)*64:,}/{len(test_dataset):,}...')
E=torch.cat(all_embs); L=torch.cat(all_lbs)
print(f'Embeddings: {E.shape}')

Ks=[1,5,10]; correct={k:0 for k in Ks}; n=E.size(0); chunk=2000
for s in range(0,n,chunk):
    e=E.size(0); end=min(s+chunk,n)
    qe=E[s:end].to(DEVICE); db=E.to(DEVICE)
    sim=torch.mm(qe,db.t())
    for i in range(end-s): sim[i,s+i]=float('-inf')
    for k in Ks:
        tl=L.to(DEVICE)[sim.topk(k,dim=1).indices]
        correct[k]+=(tl==L[s:end].to(DEVICE).unsqueeze(1)).any(1).sum().item()
    del sim,qe,db; torch.cuda.empty_cache()

final={k:correct[k]/n for k in Ks}
print(); print('='*50)
print(f'  Full Test Results (N={n:,})')
print('='*50)
for k in Ks: print(f'  Recall@{k:<2d}: {final[k]*100:.2f}%')
print('='*50)

Loading C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt ...
Loaded epoch 10 | val R@10=0.8406
Encoding all test images...
  6,400/60,502...
  12,800/60,502...
  19,200/60,502...
  25,600/60,502...
  32,000/60,502...
  38,400/60,502...
  44,800/60,502...
  51,200/60,502...
  57,600/60,502...
Embeddings: torch.Size([60502, 512])

  Full Test Results (N=60,502)
  Recall@1 : 73.23%
  Recall@5 : 83.96%
  Recall@10: 87.17%


## Cell 9 — Comparison Table vs ResNet50 Baseline

In [41]:
baseline={1:53.79,5:65.48,10:69.88}
r1=final[1]*100; r5=final[5]*100; r10=final[10]*100
print(); print('='*60)
print(f'{"Model":<16}| {"R@1":^10}| {"R@5":^10}| {"R@10":^10}')
print('-'*60)
print(f'{"ResNet50 base":<16}| {baseline[1]:>8.2f}%  | {baseline[5]:>8.2f}%  | {baseline[10]:>8.2f}%')
print(f'{"ViT ArcFace":<16}| {r1:>8.2f}%  | {r5:>8.2f}%  | {r10:>8.2f}%')
d1=r1-baseline[1]; d5=r5-baseline[5]; d10=r10-baseline[10]
fmt=lambda v: f'{"+" if v>=0 else ""}{v:.2f}%'
print(f'{"Delta":<16}| {fmt(d1):>10}| {fmt(d5):>10}| {fmt(d10):>10}')
print('='*60)
if d1>0 and d5>0 and d10>0: print('ViT ArcFace beats ResNet50 on all metrics.')
else: print('Not all metrics beat baseline yet.')


Model           |    R@1    |    R@5    |    R@10   
------------------------------------------------------------
ResNet50 base   |    53.79%  |    65.48%  |    69.88%
ViT ArcFace     |    73.23%  |    83.96%  |    87.17%
Delta           |    +19.44%|    +18.48%|    +17.29%
ViT ArcFace beats ResNet50 on all metrics.


## Cell 10 — Save Final Checkpoint and History

In [42]:
import json as _json
HIST_PATH = r'C:\visual_searh_project\checkpoints\vit_arcface_history.json'
hist_out={'final_recalls':{str(k):final[k] for k in final},
          'best_val_r10':float(ck['recall10']),'best_epoch':int(ck['epoch']),
          'baseline':{'r1':0.5379,'r5':0.6548,'r10':0.6988}}
with open(HIST_PATH,'w') as f: _json.dump(hist_out,f,indent=2)
print(f'History saved to {HIST_PATH}')
print(f'Best checkpoint: {CKPT_AF}')
print('Done.')

History saved to C:\visual_searh_project\checkpoints\vit_arcface_history.json
Best checkpoint: C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt
Done.
